# Chapter 03 Exercises

Work through each section in order. **Answer before you run the cell** — the
prediction is where the learning happens, not the output.

Sections:

- **A** — concept checks
- **B** — predict the output
- **C** — find the bug
- **D** — fix the style
- **E** — write it yourself
- **F** — self-assessment

## Section A — Concept checks

Answer each in your own words, then run the cell to compare.

In [ ]:
questions = [
    (
        "A1. What is the test for whether something is an expression?",
        "Can it go on the right-hand side of an = ? If yes, it is an "
        "expression. Expressions produce values; statements perform actions.",
    ),
    (
        "A2. Why does a bare `text.strip()` on its own line do nothing useful?",
        "strip() RETURNS a new string - it does not modify in place. As an "
        "expression statement the result is computed then discarded (POP_TOP "
        "in the bytecode). You must capture it: text = text.strip().",
    ),
    (
        "A3. What tokens does the tokenizer emit for indentation?",
        "INDENT and DEDENT. They are real tokens generated from whitespace, "
        "and they play exactly the role that { and } play in C.",
    ),
    (
        "A4. Why does Python raise TabError rather than guessing tab width?",
        "A tab's display width is not defined by the language. If tabs and "
        "spaces are mixed ambiguously, Python cannot know the nesting, so it "
        "refuses rather than silently picking the wrong structure.",
    ),
    (
        "A5. What is the difference between a comment and a docstring?",
        "A comment is discarded at compile time and cannot be read at "
        "runtime. A docstring is the first statement of a module, function or "
        "class, and is stored in __doc__ where help() and editors can read it.",
    ),
    (
        "A6. Why prefer brackets over a backslash for line continuation?",
        "A backslash breaks if a single invisible space follows it. Brackets "
        "are robust, and the tokenizer already ignores newlines inside them.",
    ),
    (
        "A7. Why is shadowing a built-in worse than using a keyword?",
        "A keyword gives an immediate SyntaxError at the point of the "
        "mistake. A built-in is not reserved, so the assignment succeeds and "
        "the failure appears later, elsewhere, with a confusing message.",
    ),
    (
        "A8. What does a double leading underscore actually do?",
        "It triggers name mangling inside a class: __x becomes _ClassName__x. "
        "It is the only underscore convention that changes behaviour, and its "
        "purpose is avoiding clashes in subclasses - not privacy.",
    ),
    (
        "A9. What does PEP 8 say about following PEP 8?",
        "Consistency within a function matters most, then the module, then "
        "the project, and consistency with PEP 8 itself comes last. Match the "
        "code you are editing.",
    ),
    (
        "A10. Why is the reported error line often not the broken line?",
        "Python reports where it NOTICED the problem. An unclosed bracket "
        "means it keeps reading, so the error surfaces on the following line. "
        "When the named line looks fine, check the line above.",
    ),
]

for question, answer in questions:
    print(question)
    print("   ", answer)
    print("")

## Section B — Predict the output

For each, write down your answer first. Then run.

In [ ]:
# B1 - expression or statement?
candidates = ["3 * 7", "x = 1", "'a' in 'cat'", "len([1,2])", "import os"]

print("B1. Which of these are expressions?")
for candidate in candidates:
    try:
        # eval() only accepts expressions.
        compile(candidate, "<demo>", "eval")
        verdict = "expression"
    except SyntaxError:
        verdict = "statement"
    print("   ", candidate.ljust(16), "->", verdict)

In [ ]:
# B2 - what does each print?
print("B2. Predict each line:")
print("")

text = "  Hello  "

# Does this change text?
text.strip()
print("   after bare text.strip():", repr(text))

# Does this?
text = text.strip()
print("   after text = text.strip():", repr(text))

print("")
print("   WHY: strings are immutable. Methods return new values.")

In [ ]:
# B3 - tuple or not?
print("B3. What type is each?")
print("")

samples = [("(5)", (5)), ("(5,)", (5,)), ("5,", 5,), ("()", ()), ("(,)", None)]

for label, value in samples[:4]:
    print("   ", label.ljust(6), "->", type(value).__name__, repr(value))

print("")
print("   WHY: the COMMA makes a tuple, not the brackets.")
print("   () is the one exception - empty brackets ARE an empty tuple.")

In [ ]:
# B4 - the missing comma.
print("B4. How many items in each list?")
print("")

correct = ["alpha", "beta", "gamma"]
broken = ["alpha", "beta" "gamma"]

print("   with all commas:  ", correct, "->", len(correct))
print("   one comma missing:", broken, "->", len(broken))
print("")
print("   WHY: adjacent string literals are joined at COMPILE time.")
print("   No error is raised. This is why linters matter.")

In [ ]:
# B5 - name mangling.
print("B5. What attribute names get stored?")
print("")

class Demo:
    def __init__(self):
        self.public = 1
        self._internal = 2
        self.__mangled = 3

instance = Demo()
for name in vars(instance):
    print("   ", name)

print("")
print("   WHY: __mangled became _Demo__mangled. Double leading underscore")
print("   triggers name mangling; single underscore is convention only.")

## Section C — Find the bug

Each cell below contains broken code. Read it, decide what is wrong and what the
error will be, **then** run it.

In [ ]:
NEWLINE = chr(10)

def check(label, source):
    """Compile a snippet and report what happens."""
    print("C" + label)
    for number, line in enumerate(source.rstrip(NEWLINE).split(NEWLINE), start=1):
        print(f"   {number} | {line}")
    try:
        compile(source, "<demo>", "exec")
        print("   -> compiles (the bug is not a syntax error)")
    except SyntaxError as error:
        print(f"   -> {type(error).__name__}: {error.msg} (line {error.lineno})")
    print("")


# C1
check("1", "for item in [1, 2, 3]" + NEWLINE + "    print(item)" + NEWLINE)

# C2
check("2", "values = [1, 2, 3" + NEWLINE + "total = sum(values)" + NEWLINE)

# C3
check("3", "if count = 5:" + NEWLINE + "    print('five')" + NEWLINE)

# C4
check("4", "def greet():" + NEWLINE + "print('hello')" + NEWLINE)

In [ ]:
# C5 - this one COMPILES but is still wrong. What is the bug?
def running_total(values):
    """Add up a list of numbers."""
    sum = 0
    for value in values:
        sum = sum + value
    return sum


print("C5. This works:", running_total([1, 2, 3]))

# But now try to use the built-in inside the same function.
def running_total_then_builtin(values):
    """The same function, then calling the built-in sum."""
    sum = 0
    for value in values:
        sum = sum + value
    # The built-in is shadowed inside this scope.
    try:
        return sum(values)
    except TypeError as error:
        return f"failed: {error}"


print("C5. But this fails:", running_total_then_builtin([1, 2, 3]))
print("")
print("   BUG: `sum` shadows the built-in. Rename it to running_total.")

## Section D — Fix the style

The code below runs correctly but violates PEP 8 in several ways. Rewrite it
yourself before revealing the fixed version.

In [ ]:
# The original, with the violations left in place as text.
original = [
    "import os,sys",
    "MAX=100",
    "def calcTotal( price,qty ):",
    "  if qty>0 :",
    "    return price*qty",
    "  else :",
    "    return 0",
    "class inventory_item :",
    "  def __init__(self,Name) :",
    "    self.Name=Name",
]

print("ORIGINAL:")
for line in original:
    print("   " + line)

print("")
print("How many PEP 8 violations can you find? (There are at least 10.)")

In [ ]:
# The fixed version. Compare against your own rewrite.
violations = [
    ("import os,sys", "one import per line"),
    ("MAX=100", "spaces around =, and a name saying what it limits"),
    ("calcTotal", "camelCase - should be snake_case"),
    ("( price,qty )", "no spaces inside brackets; space after comma"),
    ("2-space indent", "PEP 8 requires 4"),
    ("if qty>0 :", "spaces around >, no space before the colon"),
    ("else: return 0", "a guard clause reads better here"),
    ("class inventory_item", "classes use PascalCase"),
    ("def __init__(self,Name)", "space after comma; Name should be lowercase"),
    ("self.Name=Name", "attributes are snake_case; spaces around ="),
    ("no docstrings", "PEP 257 - public things get docstrings"),
    ("no blank lines", "two between top-level definitions"),
]

print("VIOLATIONS:")
for index, (what, why) in enumerate(violations, start=1):
    print(f"  {index:2}. {what.ljust(26)} {why}")

print("")
print("FIXED VERSION:")
print("")

import os
import sys

MAX_ITEMS = 100


def calculate_total(price, quantity):
    """Return the line total, or zero when nothing is ordered."""
    # Guard clause: handle the empty case and leave.
    if quantity <= 0:
        return 0

    return price * quantity


class InventoryItem:
    """A single item held in stock."""

    def __init__(self, name):
        self.name = name


print("   calculate_total(10, 3) =", calculate_total(10, 3))
print("   calculate_total(10, 0) =", calculate_total(10, 0))
print("   InventoryItem('bolt').name =", InventoryItem("bolt").name)

## Section E — Write it yourself

These need a real editor. Create each file, run it, and check the output by hand.

In [ ]:
tasks = [
    ("E1", "expressions.py",
     "Write five expressions and five statements. Prove which is which by "
     "passing each to compile(source, '<x>', 'eval') in a try/except."),
    ("E2", "nesting.py",
     "Write a function with four levels of nesting, then rewrite it using "
     "guard clauses. Confirm both give identical results for five inputs."),
    ("E3", "documented.py",
     "Write three functions with Google-style docstrings. Print each "
     "__doc__, then call help() on one."),
    ("E4", "wrapping.py",
     "Write one expression summing six values across six lines using "
     "brackets. Then a dict of six keys, one per line, with trailing commas."),
    ("E5", "naming.py",
     "Write a script using a constant, a function, a class, a private "
     "attribute and a throwaway underscore. Run ruff and fix what it finds."),
    ("E6", "errors.py",
     "Trigger all seven syntax errors from 03.7, each inside a try/except, "
     "printing the message. Then trigger five runtime errors the same way."),
]

print("Write these as real .py files:")
print("")
for number, filename, description in tasks:
    print(f"  {number}. {filename}")
    print(f"      {description}")
    print("")

print("Rules for all six:")
print("   - a comment above every meaningful line")
print("   - run `ruff check` on each and fix everything it reports")
print("   - descriptive names, no single letters")

## Section F — Self-assessment

Tick honestly. Anything you cannot tick, reread that subchapter.

In [ ]:
checklist = [
    "I can tell an expression from a statement, and explain the test.",
    "I know why a bare method call on its own line often does nothing.",
    "I can explain INDENT and DEDENT tokens.",
    "I know the difference between IndentationError and TabError.",
    "I can flatten deep nesting using guard clauses.",
    "I know why comments vanish but docstrings do not.",
    "I can write a Google-style docstring and read it back.",
    "I prefer brackets to backslashes, and know why.",
    "I know why (5) is not a tuple.",
    "I can name at least ten Python keywords from memory.",
    "I know why shadowing a built-in is worse than using a keyword.",
    "I can explain what a double leading underscore does.",
    "I know the four main PEP 8 naming conventions.",
    "I read tracebacks from the bottom up.",
    "I check the line above when the reported line looks correct.",
]

print("Ready for Chapter 04?")
print("-" * 70)
for item in checklist:
    print("   [ ]", item)

print("")
print("All ticked? Chapter 04 - Variables and Memory Model.")
print("That is where `a = b` stops being obvious.")